# IndiGo Profitability Intelligence — D5 Model Training & Evaluation

This notebook is a runnable notebook version of the received `ml_model_training(2).py` workflow. It predicts next-quarter EBITDA Margin and evaluates chronological validation and holdout performance.

**Input required:** `EDA ANALYSIS.xlsx`, worksheet `Sheet1`, with the required feature columns. Place the workbook in the notebook working directory or `/content/` in Google Colab.

**Important:** Results depend on the exact source workbook vintage. Do not interpret this notebook as a reproduction of the D5 report until the input workbook is reconciled and the run outputs match the report.

In [ ]:
# Core libraries and model dependencies
import os
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

from statsmodels.tsa.holtwinters import ExponentialSmoothing

try:
    from IPython.display import display
except ImportError:
    display = print

try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print("XGBoost is not installed; continuing without it.")

warnings.filterwarnings("ignore")



## Model training, validation, test evaluation and forecast

The following code is carried over from the received Python training script. It uses a chronological 60/20/20 split and selects the model using validation MAE.

In [ ]:
# -*- coding: utf-8 -*-
"""ML_Model_Training.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1M0O3NkEdqALqwZkeJlssEPrRhXqekhTv

# IndiGo Airlines: Profitability Intelligence & Forecasting System

**Primary objective:** Forecast next-quarter **EBITDA Margin** using historical EBITDA Margin and macroeconomic indicators (USD/INR, crude oil, CPI and WPI).

**Target:** `Target_Next_Q_EBITDA = EBITDA Margin shifted by -1 quarter`

**Evaluation:** MAE, RMSE and MASE. MASE uses a correctly aligned seasonal-naive benchmark. A MASE below 1 means the model's error is lower than the seasonal-naive benchmark error.

> **Forecast timing assumption:** This notebook assumes the forecast is made after the current quarter's EBITDA and macroeconomic data are available. If the forecast is made before quarter-end, current-quarter inputs must be replaced with values available at that forecast date.

The dataset is small (approximately 38 usable quarterly observations), so results should be treated as exploratory rather than proof of reliable future performance.

## 1. Load data and create the forecasting target

Update `DATA_PATH` if your Excel file is stored somewhere other than the notebook's working directory.
"""

# Google Colab path first; local working-directory path as fallback
possible_paths = ["/content/EDA ANALYSIS.xlsx", "EDA ANALYSIS.xlsx", "/mnt/data/EDA ANALYSIS.xlsx"]
DATA_PATH = next((path for path in possible_paths if os.path.exists(path)), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "EDA ANALYSIS.xlsx was not found. Upload it to Colab or place it beside this notebook."
    )

df = pd.read_excel(DATA_PATH, sheet_name="Sheet1")

required_cols = [
    "EBITDA Margin",
    "USD INR (Quarterly Mean)",
    "Oil benchmark(MONTHLY) (Quarterly Mean)",
    "CPI inflation (Quarterly Mean)",
    "WPI inflation (Quarterly Mean)"
]
missing = [col for col in required_cols if col not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in the Excel file: {missing}")

# Ensure rows are in chronological order before shifting.
# If the file has Year and Quarter columns, sort by them.
if {"Year", "Quarter (Calendar)"}.issubset(df.columns):
    df = df.sort_values(["Year", "Quarter (Calendar)"]).reset_index(drop=True)
else:
    df = df.reset_index(drop=True)

# One-step-ahead target: current quarter features predict next quarter EBITDA margin.
df["Target_Next_Q_EBITDA"] = df["EBITDA Margin"].shift(-1)

# Correct seasonal-naive alignment:
# For origin quarter t, predict EBITDA in t+1 using the same seasonal quarter from the prior year (t-3).
df["Seasonal_Naive_Pred"] = df["EBITDA Margin"].shift(3)

# Expanding historical mean available at the forecast origin (no future observations).
df["Historical_Mean_Pred"] = df["EBITDA Margin"].expanding().mean()

# Lagged and rolling features use current and past information only.
df["EBITDA_Lag1"] = df["EBITDA Margin"].shift(1)
df["USD_Lag1"] = df["USD INR (Quarterly Mean)"].shift(1)
df["Oil_Lag1"] = df["Oil benchmark(MONTHLY) (Quarterly Mean)"].shift(1)
df["EBITDA_2Q_Rolling"] = df["EBITDA Margin"].rolling(window=2).mean()
df["Oil_2Q_Rolling"] = df["Oil benchmark(MONTHLY) (Quarterly Mean)"].rolling(window=2).mean()

feature_cols = [
    "EBITDA Margin", "EBITDA_Lag1", "EBITDA_2Q_Rolling",
    "USD INR (Quarterly Mean)", "USD_Lag1",
    "Oil benchmark(MONTHLY) (Quarterly Mean)", "Oil_Lag1", "Oil_2Q_Rolling",
    "CPI inflation (Quarterly Mean)", "WPI inflation (Quarterly Mean)"
]

model_df = df.dropna(subset=feature_cols + ["Target_Next_Q_EBITDA", "Seasonal_Naive_Pred"]).copy()
model_df = model_df.reset_index(drop=True)

X = model_df[feature_cols]
y = model_df["Target_Next_Q_EBITDA"]

print(f"Raw observations: {len(df)}")
print(f"Usable supervised observations: {len(model_df)}")
print(f"Features: {len(feature_cols)}")
display(model_df[["Target_Next_Q_EBITDA", "Seasonal_Naive_Pred"]].head())

"""## 2. Chronological train / validation / test split

The data is not shuffled. The earliest observations train the models, the next block selects the model, and the latest block is held back for a final evaluation. The test block is not used to choose the winner.
"""

n = len(model_df)
if n < 15:
    raise ValueError("Too few usable observations for this three-way chronological split.")

train_end = int(n * 0.60)
val_end = int(n * 0.80)

X_train, y_train = X.iloc[:train_end], y.iloc[:train_end]
X_val, y_val = X.iloc[train_end:val_end], y.iloc[train_end:val_end]
X_test, y_test = X.iloc[val_end:], y.iloc[val_end:]

# Seasonal-naive and historical-mean predictions are aligned to the same forecast origins.
seasonal_val = model_df["Seasonal_Naive_Pred"].iloc[train_end:val_end].to_numpy()
seasonal_test = model_df["Seasonal_Naive_Pred"].iloc[val_end:].to_numpy()
mean_val = model_df["Historical_Mean_Pred"].iloc[train_end:val_end].to_numpy()
mean_test = model_df["Historical_Mean_Pred"].iloc[val_end:].to_numpy()

print(f"Train: {len(X_train)} | Validation: {len(X_val)} | Test: {len(X_test)}")

"""## 3. Train candidate models and select using validation data

Scaling is fitted only on the training partition through `Pipeline`, preventing information from validation or test data from affecting the scaler. ETS is included as a univariate statistical benchmark. XGBoost is optional.
"""

# Candidate feature-based models. Keep model complexity modest for this small dataset.
candidate_models = {
    "Ridge": Pipeline([("scaler", StandardScaler()), ("model", Ridge(alpha=1.0))]),
    "Lasso": Pipeline([("scaler", StandardScaler()), ("model", Lasso(alpha=0.1, max_iter=10000))]),
    "Elastic Net": Pipeline([("scaler", StandardScaler()), ("model", ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000))]),
    "Random Forest": RandomForestRegressor(n_estimators=200, max_depth=2, min_samples_leaf=2, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, max_depth=1, learning_rate=0.03, random_state=42)
}
if XGBOOST_AVAILABLE:
    candidate_models["XGBoost"] = XGBRegressor(
        n_estimators=100, max_depth=1, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.8,
        objective="reg:squarederror", random_state=42
    )

validation_predictions = {
    "Seasonal Naive": seasonal_val,
    "Historical Mean": mean_val
}
fitted_candidates = {}

for name, model in candidate_models.items():
    try:
        model.fit(X_train, y_train)
        validation_predictions[name] = np.asarray(model.predict(X_val)).reshape(-1)
        fitted_candidates[name] = model
    except Exception as exc:
        print(f"Skipping {name}: {exc}")

# ETS validation forecast. ETS uses only the training target series.
try:
    ets_model = ExponentialSmoothing(
        y_train.astype(float), trend="add", seasonal=None,
        initialization_method="estimated"
    ).fit()
    validation_predictions["ETS"] = np.asarray(ets_model.forecast(len(y_val)))
except Exception as exc:
    print(f"ETS could not be fitted: {exc}")

def regression_metrics(actual, predicted, mase_scale):
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mase = mae / mase_scale if mase_scale and mase_scale > 0 else np.nan
    return {"MAE": mae, "RMSE": rmse, "MASE": mase}

# Scale for MASE: in-sample one-year seasonal-naive error on training target series.
# This is a training-only scale and is independent of validation/test outcomes.
seasonal_errors = np.abs(y_train.to_numpy()[4:] - y_train.to_numpy()[:-4])
mase_scale = seasonal_errors.mean() if len(seasonal_errors) else np.nan

validation_rows = []
for name, pred in validation_predictions.items():
    metrics = regression_metrics(y_val, pred, mase_scale)
    validation_rows.append({"Model": name, **metrics})

validation_results = pd.DataFrame(validation_rows).sort_values("MAE").reset_index(drop=True)
display(validation_results.round(3))

# Select only from validation performance; the final test set remains untouched.
winner_name = validation_results.iloc[0]["Model"]
print(f"Selected using validation MAE: {winner_name}")

"""## 4. Refit the selected model and evaluate once on the test set

The final model is refitted on train + validation observations. The test set is used only for the final reported comparison. Baselines are included for context.
"""

X_trainval = X.iloc[:val_end]
y_trainval = y.iloc[:val_end]

if winner_name in candidate_models:
    final_model = candidate_models[winner_name]
    final_model.fit(X_trainval, y_trainval)
    test_prediction = np.asarray(final_model.predict(X_test)).reshape(-1)
elif winner_name == "ETS":
    final_model = ExponentialSmoothing(
        y_trainval.astype(float), trend="add", seasonal=None,
        initialization_method="estimated"
    ).fit()
    test_prediction = np.asarray(final_model.forecast(len(y_test)))
elif winner_name == "Seasonal Naive":
    final_model = "Seasonal Naive"
    test_prediction = seasonal_test
elif winner_name == "Historical Mean":
    final_model = "Historical Mean"
    test_prediction = mean_test
else:
    raise RuntimeError(f"Unexpected selected model: {winner_name}")

test_predictions = {
    winner_name: test_prediction,
    "Seasonal Naive": seasonal_test,
    "Historical Mean": mean_test
}

test_rows = []
for name, pred in test_predictions.items():
    metrics = regression_metrics(y_test, pred, mase_scale)
    test_rows.append({"Model": name, **metrics})

test_results = pd.DataFrame(test_rows).sort_values("MAE").reset_index(drop=True)
print("FINAL HOLDOUT TEST RESULTS")
display(test_results.round(3))

print("\nInterpretation:")
print("- Lower MAE and RMSE indicate smaller forecast errors.")
print("- MASE below 1 means the model beats the seasonal-naive training benchmark on this test period.")
print("- Because the dataset is small, performance may change materially with new quarters.")

"""## 5. Visualise the final holdout forecasts"""

plt.figure(figsize=(14, 7))
test_positions = np.arange(len(y_test))

# Plot the actual ground truth
plt.plot(test_positions, y_test.to_numpy(), marker="o", color="black", linewidth=3, label="Actual EBITDA Margin")

# Plot the baseline methods
plt.plot(test_positions, seasonal_test, marker="^", linestyle=":", color="grey", linewidth=2, label="Seasonal Naive (Baseline)")
plt.plot(test_positions, mean_test, marker="v", linestyle=":", color="silver", linewidth=2, label="Historical Mean (Baseline)")

# Generate and plot test predictions for all fitted machine learning models
colors = ["red", "blue", "green", "orange", "purple", "brown"]
for i, (name, model) in enumerate(candidate_models.items()):
    try:
        # Ensure each model is fitted on train+val before predicting on test
        model.fit(X_trainval, y_trainval)
        pred_test = np.asarray(model.predict(X_test)).reshape(-1)
        plt.plot(test_positions, pred_test, marker="s", linestyle="--", alpha=0.8, color=colors[i % len(colors)], label=f"{name}")
    except Exception as exc:
        print(f"Could not plot predictions for {name}: {exc}")

# Add ETS benchmark to the visualization as well
try:
    ets_model_full = ExponentialSmoothing(
        y_trainval.astype(float), trend="add", seasonal=None,
        initialization_method="estimated"
    ).fit()
    ets_test = np.asarray(ets_model_full.forecast(len(y_test)))
    plt.plot(test_positions, ets_test, marker="d", linestyle="-.", color="cyan", label="ETS")
except Exception as exc:
    pass

plt.title("Next-Quarter EBITDA Margin: Comparison of All Candidate Models on Holdout Test Set", fontsize=14)
plt.xlabel("Test Observation (Chronological Order)", fontsize=12)
plt.ylabel("EBITDA Margin", fontsize=12)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

"""## 6. Refit on all available observations and save the deployment artifact

After the holdout evaluation is complete, refit the selected model on all currently available historical observations for future forecasting. This full-data model is the deployment artifact; its in-sample fit is **not** a new performance evaluation.

The saved artifact contains the selected model name, fitted model, feature order and target definition. For feature-based models, the fitted pipeline includes its scaler.
"""

if winner_name in candidate_models:
    deployment_model = candidate_models[winner_name]
    deployment_model.fit(X, y)
elif winner_name == "ETS":
    deployment_model = ExponentialSmoothing(
        y.astype(float), trend="add", seasonal=None,
        initialization_method="estimated"
    ).fit()
elif winner_name == "Seasonal Naive":
    deployment_model = "Seasonal Naive"
elif winner_name == "Historical Mean":
    deployment_model = "Historical Mean"
else:
    raise RuntimeError(f"Unexpected selected model: {winner_name}")

artifact = {
    "model_name": winner_name,
    "model": deployment_model,
    "feature_columns": feature_cols,
    "target": "next_quarter_EBITDA_Margin",
    "forecast_horizon": "1 quarter",
    "data_file": os.path.basename(DATA_PATH),
    "selection_method": "lowest validation MAE using chronological split",
    "test_metrics": test_results.to_dict(orient="records")
}

MODEL_PATH = "best_ebitda_model.pkl"
with open(MODEL_PATH, "wb") as file:
    pickle.dump(artifact, file)

print(f"Saved deployment artifact: {MODEL_PATH}")
print(f"Selected model: {winner_name}")
print("Artifact includes model, feature order, target definition and holdout metrics.")

"""## 7. Generate the next-quarter forecast

The forecast below uses the latest row of engineered features. It assumes all inputs in that row are available at the forecast date. The output is a point forecast, not a guaranteed result or a statistical confidence interval.
"""

latest_features = X.iloc[[-1]]

if winner_name in candidate_models:
    next_quarter_forecast = float(deployment_model.predict(latest_features)[0])
elif winner_name == "Seasonal Naive":
    next_quarter_forecast = float(model_df["Seasonal_Naive_Pred"].iloc[-1])
elif winner_name == "Historical Mean":
    next_quarter_forecast = float(model_df["Historical_Mean_Pred"].iloc[-1])
else:
    next_quarter_forecast = float(deployment_model.forecast(1).iloc[0])

print(f"Selected model: {winner_name}")
print(f"Forecast next-quarter EBITDA Margin: {next_quarter_forecast:.2f}")
print("Note: Interpret this forecast alongside the data cutoff and latest available macroeconomic inputs.")

"""## 8. Methodology notes and limitations

1. **Target correction:** The prediction target throughout this notebook is next-quarter EBITDA Margin, not PAT Margin.
2. **Seasonal-naive alignment:** With a one-quarter-ahead target, the comparable quarter from the previous year is three rows before the current forecast origin (`shift(3)`).
3. **No random split:** Time-series observations remain in chronological order.
4. **Validation before test:** Model selection uses validation MAE. The final test block is held back for a one-time performance check.
5. **Small sample:** Approximately 38 usable observations are insufficient for strong claims about generalisation. Treat model rankings as provisional and update them as new quarters become available.
6. **Information availability:** Current-quarter financial and macro inputs are valid only if they are known at the forecast cutoff. Otherwise, lag or forecast those inputs.
7. **Not a live-data system yet:** This notebook uses a static Excel dataset. Automated data refresh, API serving, monitoring and dashboard integration are separate implementation steps.
"""
